# Dataset exploration for temporal QA

**Question:** What does the data look like, and how should it be transformed for the SELR temporal QA pipeline?

This is an exploratory, read-only notebook, not a model or benchmark. It uses the locally acquired TFLEX ICEWS14 archive, not a `train.txt` TSV. Its `Pt` records are converted into dated facts with the project's loader. The full fact graph is available to every QA split under the closed-knowledge-base policy; this notebook does not load held-out QA labels. Exploratory graph-derived examples are illustrative, not evaluation examples.

Run from the repository root or `notebooks/` with the project environment and the optional notebook dependencies documented in the README. Raw data stays local; avoid committing outputs containing source data.

In [ ]:
from collections import Counter
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import yaml
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs' / 'dataset_config.yaml').is_file()), None)
if root is None:
    raise FileNotFoundError('Run from the repository root or a subdirectory.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.graph.graph_builder import build_temporal_graph
from src.ingestion.json_loader import load_icews14_facts, load_icews14_metadata
from src.ingestion.preprocess import preprocess_facts
from src.retrieval.subgraph_retriever import retrieve_subgraph
from src.retrieval.temporal_filter import filter_facts_by_temporal_constraint

archive = root / 'data' / 'raw' / 'icews14' / 'tflex_icews14_all.zip'
metadata_path = root / 'data' / 'raw' / 'icews14' / 'tflex_icews14_meta.json'
for path in (archive, metadata_path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. See README: ICEWS14 acquisition and provenance.')
with (root / 'configs' / 'retrieval_config.yaml').open(encoding='utf-8') as file:
    retrieval_config = yaml.safe_load(file)
settings = {key: retrieval_config[key] for key in ('strategy', 'max_hops', 'max_nodes', 'max_edges')}
print('Archive:', archive)
print('Retrieval settings:', settings)

## 1. Dataset overview and transformation

The loader selects `Pt` queries from the source train, validation, and test files and expands their timestamp answers into dated facts. These source splits are **not** the project's generated QA splits. IDs are strings, labels are for display, and provenance identifies source records. Canonical preprocessing normalizes dates and IDs, removes duplicate subject/relation/object/date events, and creates stable fact IDs. It raises errors for malformed inputs.

In [ ]:
metadata = load_icews14_metadata(metadata_path)
raw_facts = load_icews14_facts(archive, metadata_path)
facts = preprocess_facts(raw_facts)
if not facts:
    raise ValueError('The ICEWS14 archive produced no canonical facts.')
raw = pd.DataFrame(raw_facts)
df = pd.DataFrame(facts)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d', errors='raise')
entity_labels = {str(key): value for key, value in metadata.entities.items()}
relation_labels = {str(key): value for key, value in metadata.relations.items()}
print('Raw shape:', raw.shape, '| raw fields:', list(raw.columns))
print('Canonical shape:', df.shape, '| canonical fields:', list(df.columns))
print('Raw timestamp example:', raw['timestamp'].iloc[0], '| parsed dtype:', df['timestamp'].dtype)
print('Source fact counts:', raw['source_split'].value_counts().to_dict())
print('Duplicate dated facts removed:', len(raw_facts) - len(facts))
display(raw[['subject_id', 'subject_label', 'relation_id', 'relation_label', 'object_id', 'object_label', 'timestamp', 'source_id']].head())
display(df.head())

## 2. Entities and relations

Counts below refer to IDs occurring in canonical facts, not necessarily every ID in the source vocabulary. Relation labels may contain inverse relations already provided by TFLEX; do not infer graph direction from labels alone.

In [ ]:
entities = set(df['subject_id']) | set(df['object_id'])
relations = set(df['relation_id'])
relation_counts = df['relation_id'].value_counts().head(20)
print(f'Entities in facts: {len(entities):,}; relations in facts: {len(relations):,}')
display(pd.DataFrame({'relation_id': relation_counts.index, 'label': [relation_labels[x] for x in relation_counts.index], 'facts': relation_counts.values}))
relation_counts.rename(index=lambda x: f'{x}: {relation_labels[x]}').sort_values().plot.barh(figsize=(10, 7), title='Top 20 relations by dated facts')
plt.xlabel('Canonical facts')
plt.tight_layout()
plt.show()

## 3. Temporal distribution

Months are grouped by year and month, so different years would not be combined. Monthly event counts show activity variation, not graph change or causal trends by themselves.

In [ ]:
monthly = df.groupby(df['timestamp'].dt.to_period('M')).size()
monthly = monthly.reindex(pd.period_range(monthly.index.min(), monthly.index.max(), freq='M'), fill_value=0)
monthly.index = monthly.index.astype(str)
print('Time span:', df['timestamp'].min().date(), 'to', df['timestamp'].max().date())
display(monthly.rename('canonical_facts').to_frame())
monthly.plot.bar(figsize=(10, 4), title='Dated facts per month')
plt.ylabel('Canonical facts')
plt.tight_layout()
plt.show()

In [ ]:
seen_pairs = set()
graph_activity = []
for month, group in df.groupby(df['timestamp'].dt.to_period('M'), sort=True):
    pairs = set(zip(group['subject_id'], group['object_id']))
    active_entities = set(group['subject_id']) | set(group['object_id'])
    graph_activity.append({'month': str(month), 'active_entities': len(active_entities), 'active_directed_pairs': len(pairs), 'new_directed_pairs': len(pairs - seen_pairs)})
    seen_pairs.update(pairs)
print('Monthly graph activity; new pairs have not occurred in an earlier observed month:')
display(pd.DataFrame(graph_activity).set_index('month').reindex(monthly.index, fill_value=0))

## 4. Graph characteristics

The pipeline uses a directed multigraph: different relations and dates between the same endpoints remain distinct. Degree counts event edges, not unique neighbors; a self-loop contributes twice to total degree. The simple directed projection below is used only for reachability and shortest paths.

In [ ]:
temporal_graph = build_temporal_graph(facts)
graph = temporal_graph.graph
simple_graph = nx.DiGraph(graph)
degrees = pd.Series(dict(graph.degree()), name='event_degree')
print(f'Directed multigraph: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} dated edges')
print('Mean event degree:', round(degrees.mean(), 2))
hubs = degrees.nlargest(10)
display(pd.DataFrame({'entity_id': hubs.index, 'label': [entity_labels[x] for x in hubs.index], 'event_degree': hubs.values}))
degrees.plot.hist(bins=50, logy=True, figsize=(9, 4), title='Total event-edge degree (log count)')
plt.xlabel('In-degree + out-degree')
plt.tight_layout()
plt.show()

## 5. Multi-hop potential

For a reproducible exploratory sample, select up to 64 entities with outgoing facts (seed 17). A two-hop opportunity means two directed edges can be followed; it does not establish a useful question or a temporally ordered sequence. The distance summary examines up to eight sampled seeds, reachable nodes only, and paths of at most three hops; it is not the graph-wide average path length.

In [ ]:
seed_pool = sorted(node for node, degree in simple_graph.out_degree() if degree)
sample_seeds = random.Random(17).sample(seed_pool, min(64, len(seed_pool)))
two_hop_seeds = [seed for seed in sample_seeds if any(simple_graph.out_degree(neighbor) for neighbor in simple_graph.successors(seed))]
distances = [distance for seed in sample_seeds[:8] for node, distance in nx.single_source_shortest_path_length(simple_graph, seed, cutoff=3).items() if node != seed]
print(f'Sampled seeds with a directed two-edge walk: {len(two_hop_seeds)}/{len(sample_seeds)}')
print('Mean reachable shortest-path distance (8 seeds, <=3 hops):', round(sum(distances) / len(distances), 3) if distances else 'No reachable nodes')
example_seed = sample_seeds[0]
print('Example seed:', example_seed, entity_labels[example_seed])
print('First 10 outgoing neighbors:', [(node, entity_labels[node]) for node in sorted(simple_graph.successors(example_seed))[:10]])

## 6. Temporal QA prototypes

The project's synthetic questions follow an ordered relation sequence from subject to object. All supporting facts must be on or before the inclusive cutoff, and **all** reachable endpoints are valid answers. The cutoff is the latest date on the illustrative witness path; it does not require the events to occur in chronological order. We use the existing retriever with graph-sized limits to enumerate prototype answers, not a single sampled endpoint.

In [ ]:
first = next((fact for fact in facts if temporal_graph.outgoing_facts(fact['object_id'])), None)
if first is None:
    raise ValueError('No directed two-hop witness exists in the canonical graph.')
second = temporal_graph.outgoing_facts(first['object_id'])[0]
prototype_cutoff = max(first['timestamp'], second['timestamp'])
wide_settings = {**settings, 'max_nodes': len(entities), 'max_edges': len(facts)}

def make_query(seed, relation_sequence, cutoff, query_id):
    return {
        'qa_id': query_id,
        'question': 'Exploratory graph-derived query',
        'seed_entity_ids': [seed],
        'relation_sequence': list(relation_sequence),
        'temporal_constraint': {'operator': 'on_or_before', 'cutoff_date': cutoff},
    }

for sequence, cutoff in [([first['relation_id']], first['timestamp']), ([first['relation_id'], second['relation_id']], prototype_cutoff)]:
    query = make_query(first['subject_id'], sequence, cutoff, f'prototype-{len(sequence)}')
    result = retrieve_subgraph(temporal_graph, query, graph_id='icews14-exploration', settings=wide_settings)
    assert not result['limits_applied']
    assert result['candidate_answer_ids']
    print(f"From {entity_labels[first['subject_id']]} via {[relation_labels[x] for x in sequence]} on or before {cutoff}:")
    print('  Answer count:', len(result['candidate_answer_ids']), '| first 10:', [(x, entity_labels[x]) for x in result['candidate_answer_ids'][:10]])
print('Two-hop witness dates:', first['timestamp'], second['timestamp'])

## 7. Retrieval feasibility

A whole incident neighborhood is not what the retriever returns. The production retriever follows outgoing edges matching a query's relation sequence, cutoff, and node/edge budgets. For each sampled seed, query its most frequent outgoing relation at the latest dataset date. Where a two-edge witness exists, also query its ordered relations. Compare configured limits with graph-sized limits on each identical query. This deliberately selected sample and late cutoff are not representative of the QA workload; truncation and candidate loss are diagnostics, not proof of model fit or supporting-path coverage.

In [ ]:
latest_cutoff = df['timestamp'].max().date().isoformat()
retrieval_rows = []
for seed in sample_seeds:
    outgoing = temporal_graph.outgoing_facts(seed)
    counts = Counter(fact['relation_id'] for fact in outgoing)
    relation = min(counts, key=lambda item: (-counts[item], item))
    sequences = [[relation]]
    witness = next((fact for fact in outgoing if temporal_graph.outgoing_facts(fact['object_id'])), None)
    if witness is not None:
        following = temporal_graph.outgoing_facts(witness['object_id'])[0]
        sequences.append([witness['relation_id'], following['relation_id']])
    for sequence in sequences:
        query = make_query(seed, sequence, latest_cutoff, f'sample-{seed}-{len(sequence)}')
        full = retrieve_subgraph(temporal_graph, query, graph_id='icews14-exploration', settings=wide_settings)
        bounded = retrieve_subgraph(temporal_graph, query, graph_id='icews14-exploration', settings=settings)
        assert not full['limits_applied']
        retrieval_rows.append({
            'seed': seed,
            'hops': len(sequence),
            'incident_facts_unfiltered': graph.in_degree(seed) + graph.out_degree(seed) - graph.number_of_edges(seed, seed),
            'unbounded_edges': len(full['facts']),
            'bounded_edges': len(bounded['facts']),
            'bounded_nodes': len(bounded['entity_id_map']),
            'limits_applied': bounded['limits_applied'],
            'candidate_loss': bool(set(full['candidate_answer_ids']) - set(bounded['candidate_answer_ids'])),
        })
retrieval_sample = pd.DataFrame(retrieval_rows)
retrieval_summary = retrieval_sample.groupby('hops').agg(queries=('seed', 'size'), median_unbounded_edges=('unbounded_edges', 'median'), median_bounded_nodes=('bounded_nodes', 'median'), truncated=('limits_applied', 'sum'), candidate_loss=('candidate_loss', 'sum'))
display(retrieval_summary)
print('Configured limits:', settings['max_nodes'], 'nodes and', settings['max_edges'], 'edges')

## 8. Temporal subgraph extraction

Compare the exact production filter with an independent date predicate. The cutoff is inclusive. A cutoff in the middle of the observed date range makes both retained and future events visible; a QA retriever additionally applies direction, relations, and bounds.

In [ ]:
dates = sorted(df['timestamp'].dt.date.unique())
cutoff = dates[len(dates) // 2].isoformat()
constraint = {'operator': 'on_or_before', 'cutoff_date': cutoff}
window = filter_facts_by_temporal_constraint(facts, constraint)
expected_ids = {fact['fact_id'] for fact in facts if fact['timestamp'] <= cutoff}
assert {fact['fact_id'] for fact in window} == expected_ids
assert all(fact['timestamp'] <= cutoff for fact in window)
print(f'Inclusive cutoff {cutoff}: {len(window):,} retained; {len(facts) - len(window):,} future facts excluded')
print('Facts on cutoff included:', sum(fact['timestamp'] == cutoff for fact in window))
display(pd.DataFrame(sorted(window, key=lambda fact: (fact['timestamp'], fact['fact_id']))[-5:]))

## 9. Embedding readiness

Canonical entity and relation IDs are mapped deterministically in sorted order. Index 0 is reserved for unknown IDs, as in the encoder; these illustrative mappings are not saved model artifacts. The encoder uses the complete graph vocabulary under the transductive policy, but fits question-token vocabulary only on training QA. Dates are encoded separately; event IDs, labels, and provenance must not be discarded from canonical facts.

In [ ]:
entity2id = {entity: index + 1 for index, entity in enumerate(sorted(entities))}
relation2id = {relation: index + 1 for index, relation in enumerate(sorted(relations))}
assert df['subject_id'].isin(entity2id).all() and df['object_id'].isin(entity2id).all()
assert df['relation_id'].isin(relation2id).all()
assert len(set(entity2id.values())) == len(entities) and len(set(relation2id.values())) == len(relations)
print('Entity embedding rows including unknown:', len(entity2id) + 1)
print('Relation embedding rows including unknown:', len(relation2id) + 1)
print('Canonical fact fields:', list(facts[0]))

## 10. MVP observations and next steps

The summary reports measured facts and bounded exploratory samples, not final benchmark results. In particular, average degree does not determine retrieval size, and these selected queries cannot establish representative supporting-path coverage or that the model will reason correctly.

In [ ]:
summary = pd.DataFrame([
    ('Canonical dated facts', f'{len(facts):,}'),
    ('Entities in facts', f'{len(entities):,}'),
    ('Relations in facts', f'{len(relations):,}'),
    ('Time span', f"{df['timestamp'].min().date()} to {df['timestamp'].max().date()}"),
    ('Mean directed multigraph event degree', f'{degrees.mean():.2f}'),
    ('Median unbounded sampled one-hop edges', f"{retrieval_summary.loc[1, 'median_unbounded_edges']:.1f}"),
    ('Median unbounded sampled two-hop edges', f"{retrieval_summary.loc[2, 'median_unbounded_edges']:.1f}" if 2 in retrieval_summary.index else 'No sampled queries'),
    ('Sampled queries truncated (all hops)', f"{retrieval_sample['limits_applied'].sum()}/{len(retrieval_sample)}"),
    ('Sampled seeds with two-edge walks', f'{len(two_hop_seeds)}/{len(sample_seeds)}'),
], columns=['Question', 'Finding'])
display(summary)
print('Conclusions:')
print('- Dated facts can be filtered at inclusive historical cutoffs; this is not forecasting.')
print(f'- The sampled one- and two-hop study observed {retrieval_sample["limits_applied"].sum()} truncated queries; assess representative QA and supporting-path coverage before choosing budgets.')
print(f'- Directed two-edge walks were observed for {len(two_hop_seeds)} of {len(sample_seeds)} sampled seeds; path existence alone does not establish benchmark difficulty.')
print('- Generate graph-verified synthetic QA, preserve provenance and split isolation, and evaluate model suitability separately.')